In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Our modules
import sys
sys.path.insert(0, '..')
import config
from src.data_loader import load_data, validate_data, get_sector_for_ticker

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Download & Load Data

Our `load_data()` function handles everything:
- Downloads adjusted close prices from yfinance
- Cleans missing values
- Caches to a parquet file so we don't re-download next time

Set `force_refresh=True` the first time to download fresh data.

In [ ]:
# First time: force_refresh=True to download
# After that: force_refresh=False (default) to use cache
prices = load_data(force_refresh=True)

In [ ]:
# Quick look at the data
print(f"Shape: {prices.shape}")
print(f"Columns: {list(prices.columns)}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nFirst 5 rows:")
prices.head()

In [ ]:
# Run validation checks
report = validate_data(prices)

print(f"Trading days: {report['trading_days']}")
print(f"Missing values: {report['missing_values']}")
print(f"\nIssues:")
if report['issues']:
    for issue in report['issues']:
        print(f"  ⚠️ {issue}")
else:
    print("  ✅ All checks passed!")

## 2. Visualize Price Histories

Let's plot the raw prices for each sector. This gives you a visual feel for:
- How different stocks in the same sector move together
- The price scale differences (why we need normalization)
- Major market events (COVID crash in Mar 2020, etc.)

In [ ]:
# Plot all stocks in each sector
for sector, tickers in config.SECTOR_TICKERS.items():
    # Only plot tickers we actually have data for
    available = [t for t in tickers if t in prices.columns]
    if not available:
        continue
    
    fig = go.Figure()
    for ticker in available:
        fig.add_trace(go.Scatter(
            x=prices.index, 
            y=prices[ticker],
            name=ticker.replace('.NS', ''),
            mode='lines'
        ))
    
    fig.update_layout(
        title=f"{sector} — Raw Prices",
        xaxis_title="Date",
        yaxis_title="Price (₹)",
        height=400,
        template='plotly_dark'
    )
    fig.show()

## 3. Normalized Prices (Base 100)

Raw prices are hard to compare because stocks trade at very different levels.
TCS might be at ₹3500 while INFY is at ₹1500 — but that doesn't mean TCS
"performed better".

**Normalization:** Set all stocks to 100 on Day 1, then track relative
performance. Now you can directly compare: "HDFC went up 40% while ICICI
went up 35%".

In [ ]:
# Normalize: divide each stock by its first price, multiply by 100
normalized = prices / prices.iloc[0] * 100

# Plot normalized prices for a specific sector (Banking)
sector = "Private Banking"
tickers = [t for t in config.SECTOR_TICKERS[sector] if t in prices.columns]

fig = go.Figure()
for ticker in tickers:
    fig.add_trace(go.Scatter(
        x=normalized.index, 
        y=normalized[ticker],
        name=ticker.replace('.NS', ''),
        mode='lines'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    title=f"{sector} — Normalized Prices (Base 100)",
    xaxis_title="Date",
    yaxis_title="Normalized Price",
    height=450,
    template='plotly_dark'
)
fig.show()

In [ ]:
# Compare raw vs log prices for one pair
stock_a = "HDFCBANK.NS"
stock_b = "ICICIBANK.NS"

fig = make_subplots(rows=2, cols=1, subplot_titles=[
    "Raw Prices",
    "Log Prices"
])

# Raw prices
fig.add_trace(go.Scatter(x=prices.index, y=prices[stock_a], name=stock_a.replace('.NS', ''), line=dict(color='#00d4aa')), row=1, col=1)
fig.add_trace(go.Scatter(x=prices.index, y=prices[stock_b], name=stock_b.replace('.NS', ''), line=dict(color='#ff6b6b')), row=1, col=1)

# Log prices
fig.add_trace(go.Scatter(x=prices.index, y=np.log(prices[stock_a]), name=f"log({stock_a.replace('.NS', '')})", line=dict(color='#00d4aa', dash='dot')), row=2, col=1)
fig.add_trace(go.Scatter(x=prices.index, y=np.log(prices[stock_b]), name=f"log({stock_b.replace('.NS', '')})", line=dict(color='#ff6b6b', dash='dot')), row=2, col=1)

fig.update_layout(height=600, template='plotly_dark', title='Raw vs Log Prices')
fig.show()

In [ ]:
# Simple spread (without hedge ratio β — see pair_selector.py for the full computation)
# For now, just log(A) - log(B) to build intuition

pairs_to_preview = [
    ("HDFCBANK.NS", "ICICIBANK.NS", "Banking"),
    ("TCS.NS", "INFY.NS", "IT"),
    ("TATASTEEL.NS", "JSWSTEEL.NS", "Metals"),
]

fig = make_subplots(rows=len(pairs_to_preview), cols=1,
                    subplot_titles=[f"{a.replace('.NS','')} vs {b.replace('.NS','')} ({s})" 
                                   for a, b, s in pairs_to_preview])

for i, (a, b, sector_name) in enumerate(pairs_to_preview, 1):
    if a in prices.columns and b in prices.columns:
        spread = np.log(prices[a]) - np.log(prices[b])
        spread_mean = spread.mean()
        
        fig.add_trace(go.Scatter(
            x=prices.index, y=spread,
            name=f"{a.replace('.NS','')}-{b.replace('.NS','')}",
            line=dict(color='#00d4aa')
        ), row=i, col=1)
        
        # Add mean line
        fig.add_hline(y=spread_mean, row=i, col=1,
                      line_dash='dash', line_color='yellow', opacity=0.5)

fig.update_layout(
    height=300 * len(pairs_to_preview), 
    template='plotly_dark',
    title='Simple Log Spreads (Preview — no hedge ratio yet)',
    showlegend=False
)
fig.show()

## 6. Daily Returns Distribution

Quick look at the return distribution. You'll notice:
- Returns are roughly bell-shaped (normal-ish)
- Log returns are more symmetric than raw returns
- There are fat tails (more extreme moves than a normal distribution predicts)

In [ ]:
# Daily log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Summary stats
print("Daily Log Return Statistics:")
print("=" * 60)
stats = log_returns.describe().T[['mean', 'std', 'min', 'max']]
stats.columns = ['Mean (%)', 'Std (%)', 'Min (%)', 'Max (%)']
# Convert to percentage for readability
stats = stats * 100
print(stats.round(3))

In [ ]:
# Correlation matrix within one sector
sector = "Private Banking"
sector_tickers = [t for t in config.SECTOR_TICKERS[sector] if t in prices.columns]
sector_returns = log_returns[sector_tickers]

# Rename for readability
sector_returns.columns = [t.replace('.NS', '') for t in sector_returns.columns]

corr_matrix = sector_returns.corr()

fig = px.imshow(
    corr_matrix, 
    text_auto='.2f',
    color_continuous_scale='RdYlGn',
    title=f'{sector} — Return Correlation Matrix',
    template='plotly_dark'
)
fig.update_layout(height=400, width=500)
fig.show()